# Introduction to fitting: from fitted parameters to derived quantities

A fit returns **parameters with an uncertainty**: the same `Parameter` objects used in the basics tutorials
on maths. This notebook fits a straight line to simulated data and then computes quantities that derive from
the fitted parameters, with their uncertainty, using ordinary arithmetic and `es.math`.

In [ ]:
import numpy as np
import plotly.graph_objects as go

import easyscience as es
from easyscience import DescriptorNumber
from easyscience import Fitter
from easyscience import Parameter
from easyscience.base_classes import ModelBase

The function `show` prints a number as value, error and unit, and, for a `Parameter` with finite bounds, the bounds.

In [ ]:
def show(number):
    """One line with value, error, unit and, for a Parameter, the finite bounds."""
    text = f'{number.value:.4f} ± {number.error or 0.0:.4f} {number.unit}'
    if hasattr(number, 'min') and (np.isfinite(number.min) or np.isfinite(number.max)):
        text += f'   in [{number.min:.4f}, {number.max:.4f}]'
    return text

In [ ]:
import base64
import contextlib
import html
import io
import re

from IPython import get_ipython
from IPython.display import HTML
from IPython.display import IFrame
from IPython.display import display
from pygments import highlight
from pygments.formatters import HtmlFormatter
from pygments.lexers import PythonLexer


def show_figure(fig: go.Figure, height: int = 420) -> None:
    """Show a Plotly figure as an embedded iframe, so that it renders in the docs."""
    page = (
        go
        .Figure(fig)
        .update_layout(template='plotly_white')
        .to_html(
            full_html=True,
            include_plotlyjs='cdn',
            default_width='100%',
            default_height=f'{height - 20}px',
        )
    )
    encoded = base64.b64encode(page.encode('utf-8')).decode('ascii')
    display(IFrame(src=f'data:text/html;base64,{encoded}', width='100%', height=height))


def solution(note: str, source: str) -> None:
    """Run a solution in a copy of the notebook namespace and show it, folded, with its output."""
    code = source.strip('\n')
    namespace = dict(get_ipython().user_ns)
    stream = io.StringIO()
    with contextlib.redirect_stdout(stream):
        exec(code, namespace)
    parts = [
        '<details class="example"><summary>Solution</summary>',
        highlight(code, PythonLexer(), HtmlFormatter()),
    ]
    result = stream.getvalue().rstrip('\n')
    if result:
        parts.append('<p>Output:</p><pre>' + html.escape(result) + '</pre>')
    if note.strip():
        text = re.sub('`([^`]+)`', r'<code>\1</code>', html.escape(note.strip()))
        parts.append('<p>' + text + '</p>')
    parts.append('</details>')
    display(HTML(''.join(parts)))


get_ipython().register_magic_function(solution, magic_kind='cell')

## The model

A model is a class that holds its fit parameters as `Parameter`s and is callable with the independent
variable. Inside the call, use plain NumPy on the `.value` of each parameter: the fitting engine calls it
many times with plain numbers. The `es.math` functions belong **after** the fit, on the fitted parameters.

In [ ]:
class StraightLine(ModelBase):
    """A straight line, y = slope * x + intercept."""

    def __init__(self, slope: float, intercept: float):
        super().__init__()
        self._slope = Parameter(name='slope', value=slope)
        self._intercept = Parameter(name='intercept', value=intercept)

    @property
    def slope(self) -> Parameter:
        return self._slope

    @property
    def intercept(self) -> Parameter:
        return self._intercept

    def __call__(self, x):
        return self._slope.value * x + self._intercept.value

## Simulated data and the fit

The data follow $y = 0.6\,x + 1.0$ with Gaussian noise of 0.3. The weights are $1/\sigma$ of each point:

In [ ]:
rng = np.random.default_rng(seed=20261013)
noise = 0.3
x = np.linspace(0.0, 10.0, 40)
y = 0.6 * x + 1.0 + rng.normal(scale=noise, size=x.size)
weights = 1.0 / np.full_like(x, noise)

The fit starts from a poor guess and finds slope and intercept. Afterwards each parameter carries its
uncertainty:

In [ ]:
model = StraightLine(slope=0.1, intercept=0.0)
fitter = Fitter(model, model)
result = fitter.fit(x, y, weights=weights)

print(show(model.slope))
print(show(model.intercept))

The chart shows the data and the fitted line. It is interactive: hover over the points to read the values.
It loads Plotly from the internet, so it needs a connection. The charts are shown with a small helper,
`show_figure(fig)`, which is defined in a hidden cell; in your own notebook `fig.show()` does the same.

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=x,
        y=y,
        mode='markers',
        name='data',
        error_y=dict(type='constant', value=noise),
    )
)
fig.add_trace(go.Scatter(x=x, y=model(x), mode='lines', name='fit'))
fig.update_layout(
    xaxis_title='x',
    yaxis_title='y',
    margin=dict(l=60, r=20, t=30, b=50),
    legend=dict(x=0.05, y=0.95),
)
show_figure(fig)

## Quantities that derive from the fit

The fitted parameters can be used like any other number with an uncertainty. A line through the origin with
the slope would reach $y = 6$ at $x = 10$:

In [ ]:
reach = model.slope * 10
print(show(reach))

The same works with the functions of `es.math`. The angle of the line with the $x$ axis is the arctangent of
the slope:

In [ ]:
print(show(es.math.arctan(model.slope, unit='deg')))

**One caution.** The slope and the intercept of one fit are correlated, but arithmetic treats numbers as
independent. A quantity that depends on **both**, such as the $x$ where the line crosses zero,
$-\text{intercept}/\text{slope}$, therefore has an error that cannot be fully trusted. A quantity that depends
on one parameter, as in the exercises below, is fine.

### Exercise 1: Angle of the line

Compute the angle of the fitted line with the $x$ axis, in **radians**, with its uncertainty.

In [ ]:
slope = model.slope

<details class="question">
<summary>Hint 1</summary>

The slope is the tangent of the angle.

</details>

<details class="question">
<summary>Hint 2</summary>

Use `es.math.arctan`; radians are the default.

</details>

In [ ]:
%%solution
angle = es.math.arctan(slope)
print(show(angle))

### Exercise 2: Inverse slope

Compute $1/\text{slope}$, the change of $x$ for a change of 1 in $y$, with its uncertainty.

In [ ]:
slope = model.slope

<details class="question">
<summary>Hint 1</summary>

A plain number can be divided by a `Parameter`.

</details>

<details class="question">
<summary>Hint 2</summary>

Write `1 / slope`.

</details>

In [ ]:
%%solution
inverse = 1 / slope
print(show(inverse))